<a href="https://colab.research.google.com/github/HELARVEGA/Lab07_Arquitectura_BERT_BETO/blob/main/Arquitectura_Transformer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# NLP + Transformers

---
## Práctica de Laboratorio 07: Inspección de Arquitectura, Capas Ocultas y Self-Attention en BERT (BETO)

**Curso:** Inteligencia Artificial  
**Modelo:** `dccuchile/bert-base-spanish-wwm-cased` (BETO)  
**Entorno:** Google Colaboratory · PyTorch + Hugging Face Transformers


---
## 1) Visualización de la Configuración de BERT 👀


In [ ]:
#!pip install transformers

In [ ]:
from transformers import BertConfig, BertModel

Cargar la configuración del modelo BERT en español

In [ ]:
config = BertConfig.from_pretrained("dccuchile/bert-base-spanish-wwm-cased")
print("Número de capas de BERT:", config.num_hidden_layers)
print("Dimensión oculta (hidden_size):", config.hidden_size)
print("Cabezas de atención:", config.num_attention_heads)
print("Capa intermedia FFN:", config.intermediate_size)
print("Tamaño del vocabulario:", config.vocab_size)

config.json:   0%|          | 0.00/648 [00:00<?, ?B/s]

Número de capas de BERT: 12
Dimensión oculta (hidden_size): 768
Cabezas de atención: 12
Capa intermedia FFN: 3072
Tamaño del vocabulario: 31002


Cargar el modelo para visualizar su arquitectura

In [ ]:
model = BertModel.from_pretrained("dccuchile/bert-base-spanish-wwm-cased", output_hidden_states=True)
print(model)

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  440MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: dccuchile/bert-base-spanish-wwm-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
pooler.dense.bias                          | MISSING    | 
pooler.dense.weight                        | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


BertModel(
  (embeddings): BertEmbeddings(
    (word_embeddings): Embedding(31002, 768, padding_idx=1)
    (position_embeddings): Embedding(512, 768)
    (token_type_embeddings): Embedding(2, 768)
    (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True, bias=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): BertEncoder(
    (layer): ModuleList(
      (0-11): 12 x BertLayer(
        (attention): BertAttention(
          (self): BertSelfAttention(
            (query): Linear(in_features=768, out_features=768, bias=True)
            (key): Linear(in_features=768, out_features=768, bias=True)
            (value): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (output): BertSelfOutput(
            (dense): Linear(in_features=768, out_features=768, bias=True)
            (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True, bias=True)
            (dropout): Dropout(p=

### 📝 Análisis Técnico — Sección 1

#### 1. Tres módulos principales de BERT

**`embeddings` (`BertEmbeddings`):**
Transforma la secuencia de IDs en vectores densos de 768 dimensiones sumando:
- `word_embeddings (Embedding(31002, 768))` — representación léxica de cada token.
- `position_embeddings (Embedding(512, 768))` — codifica la posición ordinal.
- `token_type_embeddings (Embedding(2, 768))` — distingue segmento A (0) de B (1).
Seguido de `LayerNorm` y `Dropout(p=0.1)`.

**`encoder` (`BertEncoder`):**
Pila de **12 capas `BertLayer`** idénticas que aplican autoatención multi-cabeza y una red feed-forward.

**`pooler` (`BertPooler`):**
Extrae el vector del token `[CLS]` (posición 0 de la última capa), lo proyecta con `Linear(768→768)` y aplica `Tanh`, generando una representación de frase para clasificación.

---

#### 2. Subcapas dentro de un `BertLayer`

| Subcapa | Componente | Descripción |
|---|---|---|
| Autoatención | `BertSelfAttention` | Proyecciones Q, K, V de 768→768 divididas en 12 cabezas de d_k=64 |
| Add & Norm | `BertSelfOutput` | Dense 768→768 + conexión residual + LayerNorm |
| Feed-Forward | `BertIntermediate` | Dense 768→3072 + activación GELU |
| Add & Norm | `BertOutput` | Dense 3072→768 + conexión residual + LayerNorm |

---

#### 3. Dimensión oculta (`hidden_size`)
`hidden_size = 768` se conserva en todo el flujo principal. La única expansión ocurre en `BertIntermediate` (768 → **3072**, factor ×4), que se comprime de vuelta a 768 en `BertOutput`.


---
## 2) Visualización de las Capas y Hidden States 🔍


In [ ]:
from transformers import BertTokenizer
import torch

Cargar tokenizer y modelo BERT en español

In [ ]:
tokenizer = BertTokenizer.from_pretrained("dccuchile/bert-base-spanish-wwm-cased")
model = BertModel.from_pretrained("dccuchile/bert-base-spanish-wwm-cased", output_hidden_states=True)

tokenizer_config.json:   0%|          | 0.00/364 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  440MB            

model.safetensors: downloading bytes:           |  0.00B            

vocab.txt:   0%|          | 0.00/242k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/480k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/134 [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: dccuchile/bert-base-spanish-wwm-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
pooler.dense.bias                          | MISSING    | 
pooler.dense.weight                        | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Texto de ejemplo

In [ ]:
text = "El producto de Samsung Galaxy S21 llegó el 12 de marzo y superó mis expectativas."
inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=128)
outputs = model(**inputs)

In [ ]:
print(outputs)

BaseModelOutputWithPoolingAndCrossAttentions(last_hidden_state=tensor([[[-0.3688, -0.2218, -0.6865,  ..., -0.4430,  0.2883,  0.1794],
         [-0.2616, -0.0662, -0.6243,  ..., -0.5904,  0.6507,  0.2768],
         [-0.2415, -0.2261, -0.6785,  ..., -0.1096,  0.0910,  0.5395],
         ...,
         [-1.4393, -0.1986, -0.5770,  ..., -0.5261,  0.0716,  1.0376],
         [-0.7026,  0.3509, -0.4953,  ..., -0.3322,  0.3734,  0.1284],
         [-1.2273, -0.4597, -1.0296,  ..., -0.6864,  0.9237, -0.5841]]],
       grad_fn=<NativeLayerNormBackward0>), pooler_output=tensor([[ 7.6150e-02, -4.5371e-01,  4.2192e-01,  2.5452e-01,  1.0022e-01,
         -1.6483e-01, -1.7074e-01,  3.1603e-02, -1.4617e-01, -5.2972e-01,
         -1.3702e-01, -4.8808e-01, -1.3592e-01,  5.0299e-01,  6.4649e-02,
          2.9536e-01, -4.2416e-01,  3.4857e-02, -3.2424e-02,  2.6758e-01,
         -5.7588e-01, -2.9391e-02, -2.5727e-01, -1.0941e-02, -3.1050e-01,
          2.2547e-01, -5.4086e-01,  2.6728e-01, -4.9626e-01, -2.647

Extraer los hidden states

In [ ]:
hidden_states = outputs.hidden_states
print("Número de capas (incluyendo la capa de embedding):", len(hidden_states))

Número de capas (incluyendo la capa de embedding): 13


Por ejemplo, si BERT base tiene 12 capas, se imprimirá 13 (1 de embedding + 12 codificadores)

In [ ]:
# Inspección del diccionario 'inputs' generado por el tokenizador
print("=== CONTENIDO DEL DICCIONARIO INPUTS ===")
for k, v in inputs.items():
    print(f"\nClave: '{k}' | Forma (shape): {list(v.shape)}")
    print(f"Tensor:\n{v}")

# Definición y visualización de la lista de tokens
token_list = tokenizer.convert_ids_to_tokens(inputs['input_ids'][0])
print(f"\nSecuencia de tokens desglosada ({len(token_list)} tokens):")
print(token_list)

=== CONTENIDO DEL DICCIONARIO INPUTS ===

Clave: 'input_ids' | Forma (shape): [1, 22]
Tensor:
tensor([[    4,  1162,  4196,  1008,  4262,  8643, 16685, 21291, 24010,  1057,
          8926,  4444,  1040,  1797,  1008,  3008,  1042, 27994,  1354, 12956,
          1009,     5]])

Clave: 'token_type_ids' | Forma (shape): [1, 22]
Tensor:
tensor([[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]])

Clave: 'attention_mask' | Forma (shape): [1, 22]
Tensor:
tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]])

Secuencia de tokens desglosada (22 tokens):
['[CLS]', 'El', 'producto', 'de', 'Sam', '##su', '##ng', 'Gala', '##xy', 'S', '##21', 'llegó', 'el', '12', 'de', 'marzo', 'y', 'superó', 'mis', 'expectativas', '.', '[SEP]']


In [ ]:
# Shapes de los hidden states
print("Shape de hidden_states[0]  (Embedding):", hidden_states[0].shape)
print("Shape de hidden_states[6]  (Capa 6)   :", hidden_states[6].shape)
print("Shape de hidden_states[12] (Capa 12)  :", hidden_states[12].shape)
print("Tipo de dato:", hidden_states[0].dtype)

Shape de hidden_states[0]  (Embedding): torch.Size([1, 22, 768])
Shape de hidden_states[6]  (Capa 6)   : torch.Size([1, 22, 768])
Shape de hidden_states[12] (Capa 12)  : torch.Size([1, 22, 768])
Tipo de dato: torch.float32


### 📝 Análisis Técnico — Sección 2

#### 1. Diccionario `inputs`

| Clave | Shape | Significado |
|---|---|---|
| `input_ids` | `[1, 22]` | IDs numéricos de cada token en el vocabulario de BETO (31 002 palabras). Incluye `[CLS]` al inicio e `[SEP]` al final. WordPiece fragmenta palabras compuestas: *Samsung* → `['Sam','##su','##ng']`. |
| `token_type_ids` | `[1, 22]` | Todo ceros porque se procesa **una sola oración** (Segmento A). |
| `attention_mask` | `[1, 22]` | Todo unos porque los 22 tokens son reales (sin relleno *padding*). |

---

#### 2. Por qué `len(hidden_states) == 13`

$$\text{Total} = \underbrace{1}_{\text{Embedding}} + \underbrace{12}_{\text{Capas encoder}} = 13$$

- `hidden_states[0]`: salida de `BertEmbeddings` (antes de la primera capa).  
- `hidden_states[1]`…`hidden_states[12]`: salidas sucesivas de los 12 `BertLayer`.

---

#### 3. Dimensiones de `hidden_states[0].shape = [1, 22, 768]`

| Dim | Nombre | Valor | Significado |
|---|---|---|---|
| 0 | `batch_size` | 1 | Una oración en el lote. |
| 1 | `sequence_length` | 22 | 22 tokens: `[CLS]` + 20 subtokens + `[SEP]`. |
| 2 | `hidden_size` | 768 | Dimensión vectorial continua de cada token ($d_{\text{model}}$). |

Cada capa contiene $1 \times 22 \times 768 = \mathbf{16{,}896}$ valores flotantes.


---
## 3) Diagrama Conceptual y Pseudocódigo 🙌


Pseudocódigo del mecanismo de Self-Attention

### 3.1 Diagrama Conceptual

```
Texto de entrada: "El producto de Samsung Galaxy S21 llegó..."
        │
        ▼
[ BertTokenizer ] ──► tokens WordPiece  (seq_len = 22)
        │
        ▼
[ BertEmbeddings ] ── word + position + token_type + LayerNorm
        │
        ├──────────────────────────────► hidden_states[0]  shape [1,22,768]
        ▼
┌──────────────────────────────────────────────────────┐
│ BertLayer (×12)                                      │
│  ┌────────────────────────────────┐                  │
│  │ BertSelfAttention              │                  │
│  │  Q = X·W_Q                     │                  │
│  │  K = X·W_K                     │                  │
│  │  V = X·W_V                     │                  │
│  │  Scores = Q·Kᵀ / √d_k          │                  │
│  │  Weights = softmax(Scores)     │                  │
│  │  Context = Weights · V         │                  │
│  └────────────────────────────────┘                  │
│  Add & Norm ► BertIntermediate (GELU) ► BertOutput   │
│  Add & Norm                                          │
└──────────────────────────────────────────────────────┘
        │
        ├──────────────────────────────► hidden_states[1..12]
        ▼
[ BertPooler ] ── Linear(768→768) + Tanh sobre token [CLS]
        │
        ▼
  pooler_output  shape [1, 768]
```


### 3.2 Pseudocódigo — Scaled Dot-Product Self-Attention

$$\text{Attention}(Q,K,V)=\text{softmax}\!\left(\frac{QK^T}{\sqrt{d_k}}\right)V$$

```python
# Hiperparámetros
num_heads   = 12
hidden_size = 768
d_k         = hidden_size // num_heads   # 64

# 1. Proyecciones lineales aprendidas
Q = Linear_Q(X)   # [batch, seq, 768]
K = Linear_K(X)
V = Linear_V(X)

# 2. Separar en cabezas
Q = reshape(Q, [batch, num_heads, seq, d_k])   # [1, 12, 22, 64]
K = reshape(K, [batch, num_heads, seq, d_k])
V = reshape(V, [batch, num_heads, seq, d_k])

# 3. Producto escalar
dot = matmul(Q, transpose(K, -2, -1))          # [1, 12, 22, 22]

# 4. Escalamiento por √d_k  (evita gradientes nulos)
scaled = dot / sqrt(d_k)                       # √64 = 8

# 5. Máscara de padding (opcional)
if mask is not None:
    scaled = fill(scaled, mask==0, -1e9)

# 6. Softmax → pesos de atención (suman 1 por fila)
weights = softmax(scaled, dim=-1)              # [1, 12, 22, 22]

# 7. Suma ponderada de Values
context = matmul(weights, V)                   # [1, 12, 22, 64]

# 8. Concatenar cabezas + proyección W_O
out = reshape(context, [batch, seq, hidden_size])
out = Linear_O(out)                            # [1, 22, 768]

# 9. Conexión residual + LayerNorm
output = LayerNorm(X + Dropout(out))           # [1, 22, 768]
```


In [ ]:
# Implementación funcional en PyTorch del mecanismo de Self-Attention
import torch
import torch.nn.functional as F
import math

def scaled_dot_product_attention(Q, K, V, mask=None):
    """
    Attention(Q, K, V) = softmax(Q·Kᵀ / √d_k) · V
    """
    d_k = Q.size(-1)
    scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(d_k)
    if mask is not None:
        scores = scores.masked_fill(mask == 0, -1e9)
    attention_weights = F.softmax(scores, dim=-1)
    context = torch.matmul(attention_weights, V)
    return context, attention_weights

# Parámetros reales de BETO
batch_size  = 1
seq_len     = 22
hidden_size = 768
num_heads   = 12
d_k         = hidden_size // num_heads  # 64

torch.manual_seed(42)
Q_sim = torch.randn(batch_size, seq_len, d_k)
K_sim = torch.randn(batch_size, seq_len, d_k)
V_sim = torch.randn(batch_size, seq_len, d_k)

context, weights = scaled_dot_product_attention(Q_sim, K_sim, V_sim)

print("=== VERIFICACIÓN COMPUTACIONAL ===")
print(f"Entrada  (Q, K, V)      : {list(Q_sim.shape)}  → [batch, seq_len, d_k]")
print(f"Pesos de atención        : {list(weights.shape)}  → [batch, seq_len, seq_len]")
print(f"Contexto de salida       : {list(context.shape)}  → [batch, seq_len, d_k]")
print(f"Suma softmax fila 0      : {weights[0, 0].sum().item():.6f}  (debe ser 1.0)")

=== VERIFICACIÓN COMPUTACIONAL ===
Entrada  (Q, K, V)      : [1, 22, 64]  → [batch, seq_len, d_k]
Pesos de atención        : [1, 22, 22]  → [batch, seq_len, seq_len]
Contexto de salida       : [1, 22, 64]  → [batch, seq_len, d_k]
Suma softmax fila 0      : 1.000000  (debe ser 1.0)


### 📝 Análisis Técnico — Sección 3

#### 1. Significado de cada operación

| Paso | Operación | Justificación |
|---|---|---|
| 1 | $Q,K,V = X W_Q, X W_K, X W_V$ | Tres roles semánticos: *qué busco*, *qué ofrezco*, *qué aporto*. |
| 2 | $QK^T$ | Mide compatibilidad/afinidad entre cada par de tokens. |
| 3 | $/ \sqrt{d_k}$ | Estabiliza la varianza (de $d_k$ a 1) para evitar softmax saturada. |
| 4 | $\text{softmax}$ | Convierte puntuaciones en distribución de probabilidad (suma=1 por fila). |
| 5 | $\times V$ | Pondera los vectores de valor: cada token absorbe contexto de los demás. |
| 6 | Concat + $W_O$ | Fusiona las 12 cabezas restaurando la dimensión de 768. |
| 7 | Add & Norm | Gradiente estable mediante conexión residual y normalización. |

---

#### 2. Relación con `BertSelfAttention` y justificación pedagógica

La clase **`BertSelfAttention`** (observada en `print(model)`, Sección 1) implementa exactamente los pasos anteriores:
- `query`, `key`, `value` → matrices $W_Q, W_K, W_V$
- `BertSelfOutput.dense` → proyección $W_O$

La práctica sigue una progresión **Top-Down**:
1. **Sección 1 (macro):** se ve el modelo ensamblado con sus 12 capas y 110 M de parámetros.  
2. **Sección 2 (meso):** se inspecciona el flujo tensor a tensor con datos reales.  
3. **Sección 3 (micro):** se abre el "motor" interno y se verifica su álgebra matricial.


---
## 4) Conclusiones

1. **BETO** implementa la arquitectura Transformer Encoder con 12 capas y 768 dimensiones. Su entrenamiento con *Whole Word Masking* en español le proporciona representaciones semánticas ricas del idioma.
2. Los **13 hidden states** capturan niveles de abstracción crecientes: desde representaciones léxico-posicionales (capa 0) hasta representaciones profundamente contextuales (capa 12).
3. El **mecanismo de self-attention** con escalamiento por $\sqrt{d_k}$ resuelve las limitaciones de las RNNs al permitir dependencias de largo alcance y paralelismo total en el cómputo.
